Functions for creating heatmaps and geohash maps as HTML files

In [ ]:
import pandas as pd
import folium
from folium.plugins import HeatMap
import geopandas as gpd
import geohash2
from shapely.geometry import Polygon

In [ ]:
# Function to create a heatmap
def create_heatmap(df, lat_col, lon_col):
    m = folium.Map(location=[df[lat_col].mean(), df[lon_col].mean()], zoom_start=12)
    heat_data = [[row[lat_col], row[lon_col]] for index, row in df.iterrows()]
    HeatMap(heat_data).add_to(m)
    return m

# Creating and saving heatmaps
pickup_heatmap = create_heatmap(sample_data, 'PIKCUP_LAT', 'PIKCUP_LON')
pickup_heatmap.save('/savepath/pickup_heatmap.html')

In [ ]:
geohash_kept = base_data_filtered['PIKCUP_GEOHASH'].nunique()
data = geohash_kept
# Create a DataFrame from the data
df_filtered = pd.DataFrame(data)

# Reset index to make 'PIKCUP_GEOHASH' a column
df_filtered = df_filtered.reset_index()
# Rename the 'index' column to 'PIKCUP_GEOHASH'
df_filtered = df_filtered.rename(columns={'index': 'PIKCUP_GEOHASH'})

def geohash_to_polygon(geohash):
    lat, lon, lat_err, lon_err = geohash2.decode_exactly(geohash)
    delta_lat = lat_err * 2
    delta_lon = lon_err * 2
    # Create a Shapely Polygon object from the coordinates
    return Polygon([
        (lon - delta_lon, lat - delta_lat),
        (lon - delta_lon, lat + delta_lat),
        (lon + delta_lon, lat + delta_lat),
        (lon + delta_lon, lat - delta_lat)
    ]) # Return a Polygon object instead of a list of tuples

# Create a GeoDataFrame with bounding box polygons
gdf = gpd.GeoDataFrame(df_filtered, geometry=df_filtered['PIKCUP_GEOHASH'].apply(geohash_to_polygon)) # Directly apply the modified function

# Create a folium map
m = folium.Map(location=[40.768208, -73.863510], zoom_start=10)

# Add geohash polygons to the map
for _, row in gdf.iterrows():
    polygon = row['geometry']
    folium.vector_layers.Polygon(
        locations=[(lat, lon) for lon, lat in polygon.exterior.coords],
        color='blue', weight=2, fill=True, fill_color='blue', fill_opacity=0.1
    ).add_to(m)
    folium.Marker(
        location=[polygon.centroid.y, polygon.centroid.x],
        popup=row['PIKCUP_GEOHASH']
    ).add_to(m)

# Save the map to an HTML file
m.save('/pathtosave/geohash_map.html')